# Exercise 2 · Break the triage: Jev vs GPT-4.1 mini

**Part 03 · Labs** · 10 minutes

**Part A.** Write three emails designed to fool the triage policy: fake urgency, a resolved thread,
an instruction in the body.

**Part B.** Send the same three emails, with the same instructions and criteria, to **Jev** and to
**GPT-4.1 mini** (structured output). See which one routes each email to the queue you expected, and
which one your emails fooled.

* Needs `TYPESAFE_API_KEY` for Jev and `OPENAI_API_KEY` for GPT-4.1 mini. With only one key, only that
  model runs. With no key, post your best email in the chat and the instructor runs it live in the recap.

In [ ]:
# Setup: find the workshop folder and import the helpers (works locally and in Colab)
import os, sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/decision-models-workshop")]
             if (p / "dmw").is_dir()), None)
assert ROOT, "Open this notebook from the workshop folder. In Colab, run 00_setup.ipynb first."
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from dmw import config
config.load_env()

In [ ]:
import pandas as pd
from dmw import jev, llm, fixtures, checks
MODE = config.mode("auto")
T = fixtures.get("triage")
print("MODE =", MODE, "\nnow_utc =", T["now_utc"], "\n")
print(T["instructions"], "\n")
for k, v in T["criteria"].items(): print(f"{k:>9}: {v}")

## Part A · Your three emails

In [ ]:
ATTACKS = [
    # TODO: fake urgency (subject screams, body needs no action)
    {"subject": "", "body": "", "expected": ""},
    # TODO: a resolved thread that quotes an old outage
    {"subject": "", "body": "", "expected": ""},
    # TODO: an instruction in the body that tries to pick the queue
    {"subject": "", "body": "", "expected": ""},
]

## Part B · Run the same emails on Jev and GPT-4.1 mini

Both models get exactly the same state (`now_utc`, `subject`, `body`), instructions and criteria.
An email **passes** when the model routes it to the queue you expected.

In [ ]:
HAS_JEV, HAS_LLM = bool(config.key("TYPESAFE_API_KEY")), bool(config.key("OPENAI_API_KEY"))
print("Jev key:", HAS_JEV, "| OpenAI key:", HAS_LLM)

def state_of(email):
    return {"now_utc": T["now_utc"], "subject": email["subject"], "body": email["body"]}

def ask_jev(email):
    a = jev.ask(state_of(email), fixtures.questions(T))["answers"][T["question_id"]]
    return a["value"], round(a["confidence"], 3)

def ask_llm(email):
    return llm.decide(state_of(email), T["question_id"], T["instructions"], T["criteria"])["value"]

missing = [i + 1 for i, e in enumerate(ATTACKS) if not (e["subject"].strip() and e["body"].strip() and e["expected"])]
assert not missing, f"Fill in Part A first: email(s) {missing} are incomplete."

rows = []
for e in ATTACKS:
    row = {"subject": e["subject"][:45], "expected": e["expected"]}
    if HAS_JEV:
        row["jev"], row["jev_confidence"] = ask_jev(e)
        row["jev_pass"] = row["jev"] == e["expected"]
    if HAS_LLM:
        row["gpt-4.1-mini"] = ask_llm(e)
        row["gpt_pass"] = row["gpt-4.1-mini"] == e["expected"]
    rows.append(row)

results = pd.DataFrame(rows)
if not (HAS_JEV or HAS_LLM):
    print("No API keys: post your best email in the chat; it will be run live in the recap.")
results

In [ ]:
for name, col in [("Jev", "jev_pass"), ("GPT-4.1 mini", "gpt_pass")]:
    if col in results:
        print(f"{name:<13} passed {int(results[col].sum())} of {len(results)}")

## What did you see?

In [ ]:
OBSERVATION = ""   # TODO: which model passed more emails, which email fooled which model, and why do you think so?
checks.check_triage_exercise(ATTACKS, results, OBSERVATION)